# Детекция ботов в диалогах. Бейзлайн

Постановка задачи: построение первого воспроизводимого бейзлайна на базовых признаках и валидация моделей классического бустинга.

Цели этапа:
- извлечение базовых эвристик, структурных признаков и tf-idf матриц через FeatureStore
- сохранение валидированного среза базовых признаков в аналитическое хранилище ClickHouse
- байесовский подбор гиперпараметров для LightGBM, CatBoost и XGBoost
- оценка out-of-fold качества моделей по метрике LogLoss и логирование в MLflow

In [1]:
import json

import mlflow
import numpy as np
import optuna
import polars as pl

from project.core.utils import Reproducibility
from project.features import BaselineHeuristicsView, FeatureStore, TFIDFConfig, TFIDFView
from project.infrastructure.clickhouse import FeatureDWH
from project.models.classical import (
    CatBoostTrainer,
    ClassicalTrainerConfig,
    LightGBMTrainer,
    XGBoostTrainer,
)

2026/11/09 20:16:32 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


In [2]:
Reproducibility.seed(42)

## Загрузка данных из ClickHouse DWH

выгрузим исходные диалоги и разметку обучающей и тестовой выборок из ClickHouse

In [3]:
dwh = FeatureDWH()

train_json = dwh.raw.load_dialogues_dict("train")
test_json = dwh.raw.load_dialogues_dict("test")

ytrain = dwh.raw.load_labels("train")
ytest = dwh.raw.load_labels("test")

pl.DataFrame(
    {
        "dataset": ["train", "test"],
        "dialogs_count": [len(train_json), len(test_json)],
        "participants_count": [ytrain.height, ytest.height],
    }
)

dataset,dialogs_count,participants_count
str,i64,i64
"""train""",1516,3032
"""test""",379,758


## Генерация признаков через FeatureStore

соберем базовые признаки для обучающей и тестовой выборок

In [4]:
train_dialogues = [train_json[d_id] for d_id in ytrain["dialog_id"].unique(maintain_order=True)]
test_dialogues = [test_json[d_id] for d_id in ytest["dialog_id"].unique(maintain_order=True)]

feature_store = FeatureStore(
    views=[
        BaselineHeuristicsView(),
        TFIDFView(config=TFIDFConfig(max_features=30, ngram_min=1, ngram_max=1, min_df=2, svd_components=0)),
    ]
)
feat_names = feature_store.default_features

X_tr = feature_store.fit_transform(train_dialogues)
X_te = feature_store.transform(test_dialogues)

X_tr.shape, X_te.shape

((3032, 62), (758, 62))

In [5]:
df_train_full = pl.DataFrame(X_tr, schema=feat_names).with_columns(
    [
        pl.Series("dialog_id", ytrain["dialog_id"]),
        pl.Series("participant_index", ytrain["participant_index"]),
        pl.Series("is_bot", ytrain["is_bot"]),
    ]
)
df_test_full = pl.DataFrame(X_te, schema=feat_names).with_columns(
    [
        pl.Series("dialog_id", ytest["dialog_id"]),
        pl.Series("participant_index", ytest["participant_index"]),
        pl.Series("ID", ytest["ID"]),
    ]
)
pl.DataFrame(
    {
        "dataset": ["train_full", "test_full"],
        "rows": [df_train_full.height, df_test_full.height],
        "features_total": [df_train_full.width, df_test_full.width],
    }
)

dataset,rows,features_total
str,i64,i64
"""train_full""",3032,65
"""test_full""",758,65


## Валидация качества датасетов

проверим отсутствие пропущенных значений и корректность размерностей сформированных матриц

In [6]:
pl.DataFrame(
    {
        "dataset": ["train", "test"],
        "rows": [df_train_full.height, df_test_full.height],
        "columns": [df_train_full.width, df_test_full.width],
        "null_count": [
            df_train_full.null_count().sum_horizontal()[0],
            df_test_full.null_count().sum_horizontal()[0],
        ],
    }
)

dataset,rows,columns,null_count
str,i64,i64,i64
"""train""",3032,65,0
"""test""",758,65,0


проверим распределение целевой переменной в сформированном датасете

In [7]:
df_train_full.group_by("is_bot").agg(
    count=pl.len(),
    share=(pl.len() / df_train_full.height).round(4),
).sort("is_bot")

is_bot,count,share
i64,u32,f64
0,1795,0.592
1,1237,0.408


## Регистрация среза признаков в ClickHouse DWH

зарегистрируем проверенный срез признаков в аналитическом хранилище ClickHouse

In [8]:
if not dwh.features.has_snapshot("baseline_features", "train"):
    dwh.features.save_snapshot(
        "baseline_features", df_train_full, dataset_type="train", description="Базовые эвристики реплик и диалогов"
    )
    dwh.features.save_snapshot(
        "baseline_features", df_test_full, dataset_type="test", description="Базовые эвристики реплик и диалогов"
    )

## Подготовка обучающих выборок

выделим матрицы признаков и таргеты для обучения моделей

In [9]:
feature_cols = [c for c in df_train_full.columns if c not in ("dialog_id", "participant_index", "is_bot", "ID")]
X_train = df_train_full.select(feature_cols).to_pandas()
y_train = df_train_full["is_bot"].to_numpy().astype(int)
groups_train = df_train_full["dialog_id"].to_numpy()
X_test = df_test_full.select(feature_cols).to_pandas()

pl.DataFrame(
    {
        "matrix": ["X_train", "X_test"],
        "samples": [len(X_train), len(X_test)],
        "features": [len(feature_cols), len(feature_cols)],
    }
)

matrix,samples,features
str,i64,i64
"""X_train""",3032,62
"""X_test""",758,62


проверим структуру подготовленных признаков перед запуском обучения

In [10]:
pl.DataFrame(
    {
        "feature_count": [len(feature_cols)],
        "train_bots": [int((y_train == 1).sum())],
        "train_humans": [int((y_train == 0).sum())],
        "test_samples": [len(X_test)],
    }
)

feature_count,train_bots,train_humans,test_samples
i64,i64,i64,i64
62,1237,1795,758


## Оптимизация гиперпараметров и обучение LightGBM

проведем байесовский подбор гиперпараметров в Optuna и обучим финальный ансамбль по фолдам

In [11]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [12]:
trainer_lgb = LightGBMTrainer(
    config=ClassicalTrainerConfig(
        run_name="lightgbm_baseline",
        experiment_name="bot-detection",
        description="Optuna оптимизация модели LightGBM на базовых признаках.",
        n_trials=300,
        patience=50,
        dataset_name="train_baseline",
    )
)

tuning_lgb = trainer_lgb.tune(
    X=X_train,
    y=y_train,
    groups=groups_train,
)

/home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [13]:
result_lgb = trainer_lgb.fit(
    X_train=X_train,
    y_train=y_train,
    groups=groups_train,
    X_test=X_test,
    dialog_ids_train=df_train_full["dialog_id"].to_numpy(),
    participant_indices_train=df_train_full["participant_index"].to_numpy(),
    dialog_ids_test=df_test_full["dialog_id"].to_numpy(),
    participant_indices_test=df_test_full["participant_index"].to_numpy(),
    test_ids=df_test_full["ID"].to_numpy(),
    feature_names=feature_cols,
)

🏃 View run lightgbm_baseline at: http://127.0.0.1:5000/#/experiments/7/runs/55f400a98742454a83359555d719b6ff
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


## Оптимизация гиперпараметров и обучение CatBoost

проведем байесовский подбор гиперпараметров в Optuna и обучим финальный ансамбль по фолдам

In [14]:
trainer_cb = CatBoostTrainer(
    config=ClassicalTrainerConfig(
        run_name="catboost_baseline",
        experiment_name="bot-detection",
        description="Optuna оптимизация модели CatBoost на базовых признаках.",
        n_trials=300,
        patience=50,
        dataset_name="train_baseline",
    )
)

tuning_cb = trainer_cb.tune(
    X=X_train,
    y=y_train,
    groups=groups_train,
)

/home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [15]:
result_cb = trainer_cb.fit(
    X_train=X_train,
    y_train=y_train,
    groups=groups_train,
    X_test=X_test,
    dialog_ids_train=df_train_full["dialog_id"].to_numpy(),
    participant_indices_train=df_train_full["participant_index"].to_numpy(),
    dialog_ids_test=df_test_full["dialog_id"].to_numpy(),
    participant_indices_test=df_test_full["participant_index"].to_numpy(),
    test_ids=df_test_full["ID"].to_numpy(),
    feature_names=feature_cols,
)

🏃 View run catboost_baseline at: http://127.0.0.1:5000/#/experiments/7/runs/74fc9c6e21ff48efa12ef57d0343e3d5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


## Оптимизация гиперпараметров и обучение XGBoost

проведем байесовский подбор гиперпараметров в Optuna и обучим финальный ансамбль по фолдам

In [16]:
trainer_xgb = XGBoostTrainer(
    config=ClassicalTrainerConfig(
        run_name="xgboost_baseline",
        experiment_name="bot-detection",
        description="Optuna оптимизация модели XGBoost на базовых признаках.",
        n_trials=300,
        patience=50,
        dataset_name="train_baseline",
    )
)

tuning_xgb = trainer_xgb.tune(
    X=X_train,
    y=y_train,
    groups=groups_train,
)

/home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [17]:
result_xgb = trainer_xgb.fit(
    X_train=X_train,
    y_train=y_train,
    groups=groups_train,
    X_test=X_test,
    dialog_ids_train=df_train_full["dialog_id"].to_numpy(),
    participant_indices_train=df_train_full["participant_index"].to_numpy(),
    dialog_ids_test=df_test_full["dialog_id"].to_numpy(),
    participant_indices_test=df_test_full["participant_index"].to_numpy(),
    test_ids=df_test_full["ID"].to_numpy(),
    feature_names=feature_cols,
)

🏃 View run xgboost_baseline at: http://127.0.0.1:5000/#/experiments/7/runs/a8accdf724a445efa1baebdf666ae06b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


## Сравнительная таблица бейзлайнов

извлечем метрики валидации и сравним качество базовых моделей

In [18]:
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name("bot-detection")
runs = client.search_runs(exp.experiment_id, order_by=["attributes.start_time DESC"]) if exp else []

run_names = ["catboost_baseline", "lightgbm_baseline", "xgboost_baseline"]
rows = []
for r_name in run_names:
    matched = [r for r in runs if r.data.tags.get("mlflow.runName") == r_name]
    if matched:
        m = matched[0].data.metrics
        rows.append(
            {
                "model": r_name,
                "val_logloss": round(m.get("val_logloss", 0.0), 5),
                "val_logloss_raw": round(m.get("val_logloss_raw", m.get("val_logloss", 0.0)), 5),
                "val_roc_auc": round(m.get("val_roc_auc", 0.0), 5),
                "val_accuracy": round(m.get("val_accuracy", 0.0), 5),
                "val_f1": round(m.get("val_f1", 0.0), 5),
            }
        )

pl.DataFrame(rows).sort("val_logloss")

model,val_logloss,val_logloss_raw,val_roc_auc,val_accuracy,val_f1
str,f64,f64,f64,f64,f64
"""lightgbm_baseline""",0.23017,0.23006,0.96626,0.90699,0.88347
"""xgboost_baseline""",0.23026,0.2306,0.9663,0.90633,0.88284
"""catboost_baseline""",0.23627,0.23663,0.9643,0.90501,0.88119
